# DATA 266 Homework: GAN Concepts and Autoencoders on FashionMNIST

In [ ]:
# Required personal parameters and reproducibility setup
SID4 = 2046
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print(f"SID4 = {SID4}")
print(f"SEED = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")
assert (SID4, SEED, SLICE, HP_ID, CLS_A, CLS_B) == (2046, 2046, 46, 0, 6, 3)


SID4 = 2046
SEED = 2046
SLICE = 46
HP_ID = 0
CLS_A = 6
CLS_B = 3


## 1. Fundamental Architecture of a GAN

A Generative Adversarial Network, or GAN, consists of two neural networks that are trained against each other: a **generator** and a **discriminator**.

The **generator** is responsible for creating synthetic data. It starts with a random noise vector and transforms it through a neural network to produce an output that resembles samples from the real training data. For an image dataset, for example, the generator learns to turn random noise into realistic looking images. Initially, these generated images are usually poor, but they improve as training progresses.

The **discriminator** acts as a classifier. It receives both real samples from the training dataset and fake samples produced by the generator, and tries to determine whether each sample is real or generated. Its output represents how likely it believes a sample came from the real data distribution.

During training, the two networks have opposing objectives. The discriminator is trained to correctly distinguish real samples from generated ones. The generator is trained to produce samples that are convincing enough to fool the discriminator. As the discriminator becomes better at identifying fake samples, the generator receives stronger feedback about where its generated data differs from real data and adjusts its parameters accordingly.

This creates an **adversarial training process**. The discriminator can be viewed as continuously improving its ability to detect counterfeits, while the generator continuously improves its ability to create them. Neither network is trained independently because improvement in one network changes the learning problem faced by the other.

The theoretical goal is for the generator to learn the underlying probability distribution of the real training data. In the ideal equilibrium, the generated distribution matches the real data distribution closely enough that the discriminator can no longer reliably distinguish between them. At this point, the discriminator would assign approximately a **50% probability** to whether a sample is real or generated, meaning that the generator has successfully learned to produce samples that are statistically indistinguishable from the real data.


## 2. Mode Collapse

**Mode collapse** is a common failure in GAN training where the generator learns to produce only a small variety of outputs instead of representing the full diversity of the training data. For example, if a GAN is trained on FashionMNIST, it might discover that generating a particular type of shoe consistently fools the discriminator. Instead of learning to generate shirts, dresses, bags, shoes, and other items, it may repeatedly generate similar looking shoes.

This happens because the generator is rewarded for **fooling the discriminator**, not directly for producing diverse outputs. If the generator discovers a small set of outputs that currently work well against the discriminator, it can repeatedly produce variations of those outputs. GAN training is also unstable because the generator and discriminator are continuously changing in response to each other. If one network becomes much stronger than the other, the training signal can become weak or unstable, making collapse more likely.

One technique used to reduce mode collapse is **minibatch discrimination**. A normal discriminator evaluates samples mostly independently, which can make it difficult to recognize that the generator is repeatedly producing nearly identical outputs. Minibatch discrimination allows the discriminator to compare samples within the same batch. If many generated samples are extremely similar, the discriminator can recognize this lack of diversity and classify them as fake. The generator is therefore encouraged to produce a wider variety of outputs.

Another approach is the **Wasserstein GAN, or WGAN**. The original GAN objective can provide poor or unstable gradients when the real and generated distributions have little overlap, especially early in training. WGAN replaces the original divergence based objective with the Wasserstein distance, which provides a smoother and more informative measure of how far the generated distribution is from the real distribution. This generally gives the generator more useful gradients and makes training more stable. Later approaches such as WGAN GP further improve this by using a gradient penalty to enforce the constraints required by the Wasserstein objective.

Both approaches address mode collapse differently. Minibatch discrimination directly encourages diversity by helping the discriminator detect repetitive outputs, while WGAN improves the optimization process so that the generator receives more stable feedback and is less likely to settle on only a few successful outputs.




## 3. Significant GAN Variants

#### DCGAN

**Deep Convolutional GAN, or DCGAN**, was introduced to make GANs more effective and stable when generating images. The original GAN architecture relied mainly on fully connected layers, which do not take advantage of the spatial structure present in images.

DCGAN replaces these layers with **convolutional neural networks**. The discriminator uses strided convolutions to learn spatial features from images, while the generator uses transposed convolutions to progressively transform a latent vector into an image. It also introduced architectural practices such as batch normalization and carefully chosen activation functions to improve training stability.

The main contribution of DCGAN was showing that GANs could learn useful hierarchical image features and generate significantly better images by designing the generator and discriminator specifically for visual data.


#### Conditional GAN

A standard GAN gives the user little control over what the generator produces because generation depends mainly on random noise. A **Conditional GAN, or cGAN**, addresses this limitation by giving both the generator and discriminator additional information about the desired output.

This information can be a class label or another form of context. For example, when training on FashionMNIST, instead of simply asking the generator to produce a random clothing image, we could provide the label for **Dress** and ask it to generate an image belonging specifically to that class.

The generator therefore learns a conditional distribution rather than only the overall data distribution. The discriminator also receives the condition so that it evaluates not only whether an image looks real, but whether it is consistent with the requested condition. This makes GAN generation much more controllable and is useful when specific categories or attributes need to be generated.


#### CycleGAN

**CycleGAN** addresses a different problem: translating images between two domains when paired examples are not available. Traditional supervised image translation usually requires corresponding pairs, such as the exact same scene represented in two different styles. Collecting such paired datasets can be difficult or impossible.

CycleGAN instead learns using **unpaired datasets**. It uses two generators and two discriminators. One generator learns a transformation from domain A to domain B, while the second learns the reverse transformation from B to A.

Its major innovation is **cycle consistency loss**. If an image is transformed from A to B and then transformed back from B to A, the final image should resemble the original. For example, if the model converts a horse image into a zebra and then converts that generated zebra back into a horse, the reconstructed image should remain close to the original horse.

This cycle consistency constraint prevents the generators from making arbitrary transformations while still allowing the model to learn without paired examples. As a result, CycleGAN can perform tasks such as changing seasons, transferring artistic styles, or translating between visual domains using independently collected datasets.